###Helper: move a feasible point to a vertex

In [1]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

def findvertex(A, b, z, eps=1e-8, maxsteps=100, vb=True):
    m, n = A.shape
    z = np.array(z, dtype=float)
    tr = []  # tr means tight rows (kept linearly independent)

    def addtight():
        for i in range(m):
            if i not in tr and abs(np.dot(A[i], z) - b[i]) < eps:
                if np.linalg.matrix_rank(A[tr + [i]], tol=eps) > len(tr):
                    tr.append(i)

    addtight()
    step = 0
    while len(tr) < n and step < maxsteps:
        step += 1
        # directions that keep every currently tight row tight = null space of A[tr]
        if len(tr) > 0:
            _, _, Vt = np.linalg.svd(A[tr])
            N = Vt[len(tr):].T
        else:
            N = np.eye(n)

        moved = False
        for k in range(N.shape[1]):
            for sgn in (1, -1):
                d = sgn * N[:, k]
                sl = np.inf  # sl means step length
                ni = None
                for i in range(m):
                    if i in tr:
                        continue
                    Aid = np.dot(A[i], d)
                    if Aid > eps:
                        t = (b[i] - np.dot(A[i], z)) / Aid
                        if t < sl:
                            sl = t
                            ni = i
                if ni is not None:
                    moved = True
                    break
            if moved:
                break

        if not moved:
            if vb:
                print("No vertex reachable (feasible region contains a full line).")
            break

        z = z + sl * d
        addtight()
        if vb:
            print(f"Step {step}: Hit constraint {ni}, total tight = {len(tr)}")

    if vb:
        print("\nFinal tight rows:", tr)
    return np.round(z, 8), tr

###Simplex part 2- to find optimal soln and optimal cost

In [2]:
def simplexp2(A, b, c, x0, tol=1e-9, vb=True, ms=1000): # vb means verbose, ms means max steps
    m, n = A.shape

    # move x0 to a true vertex first, so the basis (tight rows) really describes x
    x, tr = findvertex(A, b, x0, 1e-8, 100, False) # x is the current point, tr means tight rows indices
    if len(tr) < n:
        if vb:
            print("No vertex found (A must have n linearly independent rows).")
        return None, None

    if vb:
        print(f"\nInitial active constraints (basis rows): {tr}")

    I = 0 # I means iteration
    vtd = set() # vtd is visited set for bases

    while I < ms:
        I += 1
        if vb:
            print(f"\nIteration {I} :")
            print("Current x:", x.round(6))
            print("Active constraints:", tr)

        tr_tuple = tuple(sorted(tr))
        if tr_tuple in vtd:
            if vb:
                print("Cycle detected (degeneracy). Stopping via safeguard.")
            break
        vtd.add(tr_tuple)

        AB = A[tr, :]
        ABinv = np.linalg.inv(AB)

        V = -ABinv
        imp = False # imp means improved
        nextp = None  # nextp means next point
        bc = np.dot(c, x) # bc means best cost

        # Bland's rule: try directions in order of lowest constraint index (avoids cycling)
        for i in sorted(range(n), key=lambda k: tr[k]):
            v = V[:, i]
            dc = np.dot(c, v) # dc means delta cost/ change in cost
            if dc <= tol:
                continue
            # finding step length sl (ratio test over ALL rows, zero-length steps allowed)
            sl = np.inf # sl is max step length we can move along chosen direction v before hitting a new constraint
            tir = None # tir is index of row that becomes tight after moving in direction v by step sl
            for j in range(m):
                Ajv = np.dot(A[j], v)
                if Ajv > tol:
                    tj = max((b[j] - np.dot(A[j], x)) / Ajv, 0)
                    if tj < sl - tol:
                        sl = tj
                        tir = j

            if tir is None:
                if vb:
                    print(f"Unbounded direction found (v_{i}). Problem is unbounded.")
                return x, np.inf

            pp = x + sl * v  # pp means potential point
            pc = np.dot(c, pp) # pc means potential cost
            imp = True
            bc = pc
            nextp = pp
            rir = i  # rir is the row index in the current tight set, which is to be replaced
            ni = tir # ni is the index of new constraint that became tight after moving to new vertex
            break

        if not imp:
            if vb:
                print("No improving direction, so, optimum reached.")
            break

        x = nextp
        tr[rir] = ni

        if vb:
            print(f"→ Moving to new vertex: {x.round(6)}")
            print(f"Objective value: {bc:.6f}")

    return x, np.dot(c, x)

###Simplex part 1- to find initial feasible point if not given

In [3]:
def simplexp1(A, b, tol=1e-9, vb=True):
    m, n = A.shape

    # Phase I: auxiliary LP in variables (x, s):
    #   maximize -s   subject to   A x - s <= b,   -s <= 0
    # The original problem is feasible iff the optimum is s = 0.
    # (A and b are NOT modified.)
    Aa = np.zeros((m + 1, n + 1)) # Aa means auxiliary A
    Aa[:m, :n] = A
    Aa[:m, n] = -1
    Aa[m, n] = -1
    ba = np.append(b, 0.0).astype(float) # ba means auxiliary b
    ca = np.zeros(n + 1) # ca means auxiliary cost
    ca[n] = -1

    # x = 0 with large enough s is always feasible for the auxiliary LP
    y0 = np.zeros(n + 1)
    y0[n] = max(0.0, -np.min(b))

    y, val = simplexp2(Aa, ba, ca, y0, tol, False)
    if y is None or val < -1e-7:
        print("Phase I failed: no feasible x0 found.")
        return None

    xls = y[:n]
    if np.any(A @ xls - b > 1e-7):
        print("Phase I failed: no feasible x0 found.")
        return None

    if vb:
        print("Feasible x0 found:", xls.round(6))
    return xls

###Simplex with checks

In [4]:
def simplex(A, b, c, x0=None, tol=1e-9, vb=True):
    if x0 is None:
        x0 = simplexp1(A, b, tol, vb)
        if x0 is None:
            print("Problem infeasible — cannot proceed to Phase II.")
            return None, None
    return simplexp2(A, b, c, x0, tol, vb)

###Reading input and displaying result

In [5]:
if __name__ == "__main__":
    file_name = "testcase_4.csv"
    data = pd.read_csv(file_name, header=None).to_numpy()

    z = data[0, :-1] # Initial feasible point (not used: Phase I finds its own)
    c = data[1, :-1]   # Cost vector
    A = data[2:, :-1] # Constraint matrix
    b = data[2:, -1]   # Constraint vector
    print("Cost vector:", c)
    print("Constraint matrix:")
    print(A)
    print("Constraint vector:", b)

    xopt, val = simplex(A, b, c, x0=None)

    if xopt is not None:
        if np.isinf(val):
            print("\nProblem is unbounded (objective = inf). Last vertex visited:", xopt)
        else:
            print("\nOptimal solution:", xopt)
            print("Optimal objective value:", val)

Cost vector: [3 9]
Constraint matrix:
[[ 0 -1]
 [-1  0]
 [-1  4]
 [-1  2]]
Constraint vector: [0 0 8 4]
Feasible x0 found: [0. 0.]

Initial active constraints (basis rows): [0, 1]

Iteration 1 :
Current x: [0. 0.]
Active constraints: [0, 1]
→ Moving to new vertex: [0. 2.]
Objective value: 18.000000

Iteration 2 :
Current x: [0. 2.]
Active constraints: [2, 1]
Unbounded direction found (v_1). Problem is unbounded.

Problem is unbounded (objective = inf). Last vertex visited: [0. 2.]
